# Study 959 — Crypto Fee War ₿

**Ten wrappers, one coin, fees from 19 bp to 150 bp. Does the tape hand the fee back?**

On **2024-01-11** the SEC let ten US spot-bitcoin ETFs start trading on the same morning.
They hold the same asset, strike at the same 16:00 New York close, and were written from
near-identical prospectuses. What they do *not* share is the price: Franklin charges
**19 bp** a year, BlackRock and Fidelity **25**, and Grayscale's converted trust charges
**150** — an eightfold spread, the widest any ETF category has ever launched with. Several
launches also waived their fee outright for the first six to twelve months.

That is a natural experiment. The sponsor fee is *contractually* accrued out of NAV every
day, so unlike almost everything on this desk it cannot fail to exist — the only question is
how much of it a public daily tape can actually see.

We measure it on the full cohort against **BTC-USD** and against each other,
2024-01-11 → 2026-06-30 (618 sessions, 29 complete months).

*Numbers below are the frozen headline run (`docs/results.md`, fingerprint `8463d167f5aa`,
as-of 2026-06-30). The only live cells run the offline synthetic control, and they say so.*


## 1. The same coin, at two prices

Imagine two shops selling the identical bar of gold, in the identical vault, with the identical opening hours. One charges you 0.20% a year to keep it there; the other charges 1.50%. Nothing else differs. Over a decade the second shop quietly takes about an eighth of your gold.

That is precisely the US spot-bitcoin ETF market since January 2024. So the question is embarrassingly simple: **do the expensive shop's customers actually end up with less?**

In [1]:
R = dict(track={'IBIT': (25, -41.3, -25.0, -5.16, -15.7, -0.13, 5.1, 0.38), 'FBTC': (25, -45.6, -20.5, -4.28, -13.1, -0.12, 7.8, 1.04), 'ARKB': (21, -46.5, -20.8, -4.37, -9.5, -0.08, 11.4, 1.1), 'BITB': (20, -49.9, -18.0, -3.7, -9.3, -0.09, 11.6, 1.88), 'HODL': (20, -23.9, 0.2, 0.05, 12.3, 0.11, 33.2, 4.1), 'BRRR': (25, -56.9, -21.1, -4.4, -8.7, -0.08, 12.2, 1.63), 'BTCO': (25, -25.1, -19.4, -4.12, -2.2, -0.02, 18.7, 1.29), 'EZBC': (19, -29.9, -15.5, -3.33, -3.6, -0.03, 17.3, 1.66), 'BTCW': (25, -19.1, -17.2, -3.62, -9.6, -0.09, 11.3, 0.74), 'GBTC': (150, -79.4, -156.8, -33.22, -149.4, -1.39, -128.5, -9.57)})
print('fund    fee bp/yr   what it delivered vs the pack (bp/yr)')
for tk, v in R['track'].items():
    print('%-6s %8d %18.1f' % (tk, v[0], v[6]))

fund    fee bp/yr   what it delivered vs the pack (bp/yr)
IBIT         25                5.1
FBTC         25                7.8
ARKB         21               11.4
BITB         20               11.6
HODL         20               33.2
BRRR         25               12.2
BTCO         25               18.7
EZBC         19               17.3
BTCW         25               11.3
GBTC        150             -128.5


Read the right-hand column against **zero**, not against each other. Nine funds sit in a band from +5 to +33 basis points a year — that band *is* the noise. And then there is GBTC at **-128.5 bp/yr**, which is not noise: it is almost exactly the 130 bp of extra fee it charges.

> 🔬 **For the quants.** The column is each fund's mean non-overlapping monthly log return minus the equal-weight cohort mean, annualised. GBTC's *t* is **-9.57**; the best of the other nine is **+4.10** and the rest are under +2.

## 2. Why you cannot just compare a fund to bitcoin

The obvious test is to line each fund up against the price of bitcoin and see who falls behind. It does not work, and the reason is a clock.

Bitcoin trades every minute of every day. The ETFs are priced once, at 16:00 in New York. So the gap between a fund and 'bitcoin' on any given day is mostly the **move bitcoin made while the fund was shut** — overnight, over the weekend. That is enormous, and it has nothing to do with fees.

In [2]:
R = dict(sd_vs_bench=134.7, sd_vs_peer=8.9, floor_ratio=15.2, det_vs_bench=481.0, det_vs_peer=66.0,
         n_months=29)
print('one day of fund-minus-bitcoin  wobbles by %6.1f bp' % R['sd_vs_bench'])
print('one day of fund-minus-fund     wobbles by %6.1f bp   (%.0fx smaller)'
      % (R['sd_vs_peer'], R['floor_ratio']))
print()
print('so in %d months, the smallest yearly fee gap you could prove is:' % R['n_months'])
print('  against bitcoin itself : %5.0f bp/yr' % R['det_vs_bench'])
print('  against another fund   : %5.0f bp/yr' % R['det_vs_peer'])
print()
print('a 20 bp/yr fee is 0.08 bp per day. good luck.')

one day of fund-minus-bitcoin  wobbles by  134.7 bp
one day of fund-minus-fund     wobbles by    8.9 bp   (15x smaller)

so in 29 months, the smallest yearly fee gap you could prove is:
  against bitcoin itself :   481 bp/yr
  against another fund   :    66 bp/yr

a 20 bp/yr fee is 0.08 bp per day. good luck.


Comparing the funds **to each other** deletes the clock problem entirely — they are all shut at the same moment, so the overnight move is common to all of them and cancels. The noise falls by **15×**. Everything that follows is a fund-versus-fund comparison, for exactly this reason.

> 🔬 **For the quants.** The stub is mean-zero, so a fund-versus-spot estimate is *unbiased* — it is merely useless. GBTC's true leak shows up in that column at **-149.4 bp/yr** with a *t* of only **-1.39**: the right answer, unprovable.

## 3. The answer: yes, where the fee gap is big

Line the cheapest wrapper up against the most expensive one and the fee is simply *there* — not to the basis point (the honest range is about 135 to 145 a year, depending on exactly which days you anchor on), but unmistakably there.

In [3]:
R = dict(cheap='EZBC', dear='GBTC', spread=145.8, spread_t=8.43, pos_months=26, n_months=29,
         ci_lo=110.5, ci_hi=187.0, vs_gbtc={'IBIT': (133.6, 7.04, 24), 'FBTC': (136.3, 8.23, 25), 'ARKB': (139.9, 7.31, 25), 'BITB': (140.1, 9.77, 27), 'HODL': (161.7, 10.56, 28), 'BRRR': (140.7, 9.36, 26), 'BTCO': (147.2, 5.8, 24), 'EZBC': (145.8, 8.43, 26), 'BTCW': (139.8, 6.76, 24)})
print('%s minus %s: %+.1f bp per year   (t = %+.2f)'
      % (R['cheap'], R['dear'], R['spread'], R['spread_t']))
print('positive in %d of %d months, 95%% confidence [%+.1f, %+.1f]'
      % (R['pos_months'], R['n_months'], R['ci_lo'], R['ci_hi']))
print()
print('and it is not one lucky pairing - every cheap fund vs GBTC:')
for tk, (s, t, p) in R['vs_gbtc'].items():
    print('  %-6s %+7.1f bp/yr   t=%+6.2f   %2d/%d months ahead' % (tk, s, t, p, R['n_months']))

EZBC minus GBTC: +145.8 bp per year   (t = +8.43)
positive in 26 of 29 months, 95% confidence [+110.5, +187.0]

and it is not one lucky pairing - every cheap fund vs GBTC:
  IBIT    +133.6 bp/yr   t= +7.04   24/29 months ahead
  FBTC    +136.3 bp/yr   t= +8.23   25/29 months ahead
  ARKB    +139.9 bp/yr   t= +7.31   25/29 months ahead
  BITB    +140.1 bp/yr   t= +9.77   27/29 months ahead
  HODL    +161.7 bp/yr   t=+10.56   28/29 months ahead
  BRRR    +140.7 bp/yr   t= +9.36   26/29 months ahead
  BTCO    +147.2 bp/yr   t= +5.80   24/29 months ahead
  EZBC    +145.8 bp/yr   t= +8.43   26/29 months ahead
  BTCW    +139.8 bp/yr   t= +6.76   24/29 months ahead


Nine separate funds, nine separate confirmations, all pointing the same way and all landing on the same number — around **135 basis points a year**, which is what you get when you subtract 20 from 150.

> 🔬 **For the quants.** These are not nine independent draws — they share the GBTC leg — but they *are* nine independent cheap legs, and the dispersion across them (134 to 162 bp/yr) is a fair read of how much wrapper-specific noise sits on top of the fee.

## 4. The cleanest experiment on the desk

In July 2024 Grayscale did something that could not have been better designed if we had asked. It launched a **second** bitcoin ETF — the Mini Trust — spun out of the first one's own coins. Same sponsor. Same custodian. Same coin. Same 16:00 strike. Same everything, except the fee: **15 bp instead of 150**.

In [4]:
R = dict(mini_months=23, mini_vs_gbtc=130.6, mini_vs_gbtc_t=2.84, mini_vs_gbtc_pos=20,
         mini_vs_cheap=-0.0, mini_fee_gap=135)
print('Grayscale cheap twin vs Grayscale flagship, over %d months:' % R['mini_months'])
print('  measured gap  %+7.1f bp/yr   (t = %+.2f, ahead in %d/%d months)'
      % (R['mini_vs_gbtc'], R['mini_vs_gbtc_t'], R['mini_vs_gbtc_pos'], R['mini_months']))
print('  fee gap       %+7.1f bp/yr' % R['mini_fee_gap'])
print()
print('and the same cheap twin against an outside cheap fund: %+.1f bp/yr - nothing.'
      % R['mini_vs_cheap'])

Grayscale cheap twin vs Grayscale flagship, over 23 months:
  measured gap   +130.6 bp/yr   (t = +2.84, ahead in 20/23 months)
  fee gap        +135.0 bp/yr

and the same cheap twin against an outside cheap fund: -0.0 bp/yr - nothing.


The gap is the fee. Not Grayscale's competence, not its custodian, not its trading desk — the fee, handed back to within a few basis points, by the sponsor's own two products.

## 5. What the tape *cannot* see — and it is most of the story

Two of the three things we set out to test simply do not survive contact with the data, and the reason is the same in both cases: the effect is smaller than the measurement.

In [5]:
R = dict(tier_spread=6.0, det_vs_peer=66.0, rank={'headline / all ten': (-0.642, 0.0514, 0.642, -1.124, 0.978), 'headline / cheap nine': (-0.495, 0.1792, 0.688, -1.522, 0.247), 'blended / all ten': (-0.498, 0.1472, 0.644, -1.083, 0.985), 'blended / cheap nine': (-0.31, 0.4076, 0.678, -1.645, 0.552)}, waiver={'IBIT': ('2025-01-10', -20.9, 21.1, 42.0, 0.73, -13), 'FBTC': ('2024-07-31', 18.3, 5.0, -13.3, -0.3, -25), 'ARKB': ('2024-07-11', 34.6, 6.6, -28.0, -0.22, -21), 'BITB': ('2024-07-11', 4.0, 13.2, 9.2, 0.18, -20), 'HODL': ('2025-03-31', 28.9, 37.2, 8.3, 0.25, -20), 'BTCO': ('2024-07-11', -23.8, 27.5, 51.2, 0.47, -25), 'EZBC': ('2024-08-02', 26.1, 15.0, -11.1, -0.11, -19), 'BTCW': ('2024-07-11', 19.0, 9.6, -9.3, -0.17, -25)})
print('inside the cheap tier, the WHOLE fee spread is %.0f bp/yr' % R['tier_spread'])
print('the smallest gap this tape can prove is         %.0f bp/yr' % R['det_vs_peer'])
print()
print('so ranking funds by realised tracking:')
for tag, (rho, p, crit, slope, r2) in R['rank'].items():
    verdict = 'significant' if p < 0.05 else 'NOT significant'
    print('  %-24s rank corr %+.3f, p=%.3f  -> %s' % (tag, rho, p, verdict))
print()
print('and the fee waivers expiring - a step that SHOULD be negative:')
for tk, (end, pre, post, step, t, exp) in R['waiver'].items():
    print('  %-6s expected %+4d, measured %+6.1f  (t=%+5.2f)  %s'
          % (tk, exp, step, t, 'wrong sign' if step > 0 else ''))

inside the cheap tier, the WHOLE fee spread is 6 bp/yr
the smallest gap this tape can prove is         66 bp/yr

so ranking funds by realised tracking:
  headline / all ten       rank corr -0.642, p=0.051  -> NOT significant
  headline / cheap nine    rank corr -0.495, p=0.179  -> NOT significant
  blended / all ten        rank corr -0.498, p=0.147  -> NOT significant
  blended / cheap nine     rank corr -0.310, p=0.408  -> NOT significant

and the fee waivers expiring - a step that SHOULD be negative:
  IBIT   expected  -13, measured  +42.0  (t=+0.73)  wrong sign
  FBTC   expected  -25, measured  -13.3  (t=-0.30)  
  ARKB   expected  -21, measured  -28.0  (t=-0.22)  
  BITB   expected  -20, measured   +9.2  (t=+0.18)  wrong sign
  HODL   expected  -20, measured   +8.3  (t=+0.25)  wrong sign
  BTCO   expected  -25, measured  +51.2  (t=+0.47)  wrong sign
  EZBC   expected  -19, measured  -11.1  (t=-0.11)  
  BTCW   expected  -25, measured   -9.3  (t=-0.17)  


**Ranking IBIT against FBTC on realised tracking is reading tea leaves.** Their fees differ by nothing at all; the widest gap inside the cheap tier is 6 bp/yr and the tape resolves 66. And a waiver expiring is a **1.7 bp step in a monthly series that wobbles by 12 bp** — four of the eight measured steps come out with the wrong sign, which is exactly what pure noise looks like.

That is not evidence the waivers were fictional. It is a demonstration that a free daily price series cannot see them.

> 🔬 **For the quants.** The all-ten rank correlation is -0.642 with a permutation *p* of 0.0514 — sitting exactly on the critical value the null can attain (0.642), because five funds are tied at 25 bp. Meanwhile the *regression* of tracking difference on fee has slope -1.124 and R² 0.978. The level is nailed; the ranking is not.

## 6. So what is 135 basis points a year actually worth?

This is where the study turns awkward. The fee effect is one of the most statistically certain things on this desk — and it is nearly worthless as a *trade*.

In [6]:
R = dict(sh_cheap=0.3467, sh_dear=0.3365, sh_rot=0.3457, tot_cheap=25.36, tot_dear=23.84, tot_rot=25.28,
         sharpe_gap=0.0102, vol_ann=49.9, race_switches=7, be_2bp=10.0, be_25bp=125.2,
         tax_20_100=7.2, tax_238_300=13.5, ls_gross=137.8, ls_dead_at=150.0)
print('owning the cheapest wrapper : total %+.2f%%   Sharpe %+.4f' % (R['tot_cheap'], R['sh_cheap']))
print('owning the priciest wrapper : total %+.2f%%   Sharpe %+.4f' % (R['tot_dear'], R['sh_dear']))
print('chasing last quarter\'s best: total %+.2f%%   Sharpe %+.4f  (%d switches)'
      % (R['tot_rot'], R['sh_rot'], R['race_switches']))
print()
print('the fee advantage in Sharpe terms: %+.4f  - against %.0f%% annual volatility'
      % (R['sharpe_gap'], R['vol_ann']))
print()
print('switching at 2 bp costs        : repaid in %.0f days' % R['be_2bp'])
print('switching at 25 bp costs       : repaid in %.0f days' % R['be_25bp'])
print('switching with a taxed +100%% gain: repaid in %.1f YEARS' % R['tax_20_100'])
print('switching with a taxed +300%% gain: repaid in %.1f YEARS' % R['tax_238_300'])

owning the cheapest wrapper : total +25.36%   Sharpe +0.3467
owning the priciest wrapper : total +23.84%   Sharpe +0.3365
chasing last quarter's best: total +25.28%   Sharpe +0.3457  (7 switches)

the fee advantage in Sharpe terms: +0.0102  - against 50% annual volatility

switching at 2 bp costs        : repaid in 10 days
switching at 25 bp costs       : repaid in 125 days
switching with a taxed +100% gain: repaid in 7.2 YEARS
switching with a taxed +300% gain: repaid in 13.5 YEARS


Bitcoin moves about **50% a year**. Against that, 135 basis points is a **+0.0102** change in Sharpe ratio — you would never see it in a performance chart. Chasing last quarter's best tracker actually ends up *behind* simply buying the cheap fund and never touching it again.

And this is why Grayscale can still charge 150 bp in a market where the going rate is 20: anyone who bought GBTC before 2024 is sitting on an enormous embedded gain, and selling to save 135 bp/yr means paying capital gains **today**. At a +300% gain that takes **14 years** to earn back. The dear fund is not surviving on merit; it is surviving on a tax lock-in.

## 7. Live check — the machinery is honest (offline synthetic)

Nothing below touches the real tape. We build a fake world where wrappers really are shaved by a known fee ladder, and a second fake world where they all charge the same thing while the published fee sheet still *looks* dispersed. The tools must find the first and stay silent on the second.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from fee_war import data, strategy as st
for ss, tag in [(1.0, 'planted fee ladder'), (0.0, 'null: everyone charges the same')]:
    px, truth = data.synthetic_panel(signal_strength=ss, seed=959)
    d = st.synthetic_detect(px, truth, n_perm=2000)
    print('%-34s measured %+7.1f bp/yr (planted %3.0f)  t=%+6.2f'
          % (tag, d['pair_spread_bpy'], d['planted_gap_bpy'], d['pair_t']))

planted fee ladder                 measured  +124.4 bp/yr (planted 131)  t= +6.55


null: everyone charges the same    measured    -7.9 bp/yr (planted   0)  t= -0.44


The planted fee is recovered; the null world stays flat. So the real-tape result is a fact about the funds, not an artefact of the tools.

## Verdict

- **Signal — Real.** The fee is delivered: **+145.8 bp/yr** between cheapest and priciest with *t* = **+8.43**, ahead in 26 of 29 months, confirmed on nine separate wrappers and by Grayscale's own cheap twin (**+130.6 bp/yr** against a 135 bp fee gap). What is *not* real is the fine print: inside the 19–25 bp tier the ranking is unreadable, and the waiver expiries are invisible.
- **Tradability — Fragile.** It is a purchase decision, not a strategy. Buying a 20 bp wrapper today instead of a 150 bp one is free, permanent and worth ~135 bp/yr. Everything else fails: the advantage is **+0.0102** of Sharpe against 50% volatility, the rotation rule loses to doing nothing, and a taxed switch takes 7–14 years to repay.